# Подбор параметров для PSD

## Импорты

In [1]:
from itertools import product

import pandas as pd
from tqdm import tqdm

from src.signal_processing.feature_engineering import calc_psd_at_step_offset
from src.signal_processing.metrics import eval_metrics

## Загрузка данных

In [2]:
signal_df = pd.read_parquet("../data/Run200_Wave_0_1_base_features.parquet")

## Подбор параметров

In [3]:
offsets = range(6)
gates = range(5, 11)
tasks = list(product(offsets, gates))
metric_cols = ['fom', 'silhouette_score', 'calinski_harabasz_score']
result = []

In [4]:
def eval_for_offset_gate(df, offset_ticks, short_gate):
    arr = calc_psd_at_step_offset(df, offset_ticks=offset_ticks, short_gate=short_gate)['psd'].to_numpy()
    return {k: v for k, v in eval_metrics(arr).items() if k in metric_cols}

In [5]:
for offset, gate in tqdm(tasks):
    acc = {"offset": offset, "gate": gate}
    metrics = eval_for_offset_gate(signal_df, offset_ticks=offset, short_gate=gate)
    acc['fom'] = metrics['fom']
    acc['silhouette_score'] = metrics['silhouette_score']
    acc['calinski_harabasz_score'] = metrics['calinski_harabasz_score']
    result.append(acc)

100%|██████████| 36/36 [10:03<00:00, 16.75s/it]


In [7]:
psd_df = pd.DataFrame(result)
psd_df.head()

,offset,gate,fom,silhouette_score,calinski_harabasz_score
0,0,5,1.147567,0.787785,171279.460695
1,0,6,1.112106,0.774987,153637.749594
2,0,7,1.065691,0.766538,144641.549065
3,0,8,1.024288,0.755772,134096.296664
4,0,9,0.990492,0.754857,133278.593347


In [8]:
psd_df

,offset,gate,fom,silhouette_score,calinski_harabasz_score
0,0,5,1.147567,0.787785,171279.460695
1,0,6,1.112106,0.774987,153637.749594
2,0,7,1.065691,0.766538,144641.549065
3,0,8,1.024288,0.755772,134096.296664
4,0,9,0.990492,0.754857,133278.593347
5,0,10,0.964540,0.741381,121321.451459
6,1,5,1.082490,0.772776,154303.836963
7,1,6,1.035528,0.753806,134228.162777
8,1,7,0.996026,0.753982,134231.488817
9,1,8,0.963995,0.745674,126108.119969


Самые лучшие значения offset_ticks=0, short_gate=5. 

В такой конфигурации при переборе параметров получены следующие метрики:
- Figure of Merit = 1.147567
- Silhouette Score = 0.787785
- Calinski-Harabasz Score = 171279.46

Calinski-Harabasz Score оказался слишком чувствительным к таким данным, поэтому будем ориентироваться на FOM и Silhouette. 
